<a href="https://colab.research.google.com/github/balloontip/deep-learning/blob/main/chapter-08/08-03-RNN-Sequence-to-One-Prediction-Model.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Build a simple recurrent neural network in PyTorch for sequence-to-one prediction. This notebook demonstrates RNN parameter initialization, hidden-state initialization, sequence processing, tensor shapes, extraction of the final time-step representation, and mapping the final hidden output to a prediction through a fully connected layer.

In [1]:
import torch
import torch.nn as nn


class SimpleRNNPredictor(nn.Module):
    """
    A simple RNN model designed for sequence-to-one prediction tasks.
    It takes an input sequence and predicts a single output at the end.
    """

    def __init__(self, input_size, hidden_size, output_size):
        """
        Initializes the SimpleRNNPredictor.

        Args:
            input_size (int): The number of features in each time step of the input sequence.
            hidden_size (int): The number of features in the RNN's hidden state.
            output_size (int): The number of features in the final output.
        """
        # Call the parent class's constructor to set up the module.
        super().__init__()

        # Store hidden_size for initializing the hidden state later.
        self.hidden_size = hidden_size

        # The main RNN layer. batch_first=True makes input and output tensors
        # have the shape (batch_size, sequence_length, features), which is more intuitive.
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)

        # A fully connected (linear) layer for the final output.
        # This maps the hidden state's features to the desired output size.
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        """
        Performs the forward pass of the model.

        Args:
            x (torch.Tensor): The input tensor of shape
                              (batch_size, sequence_length, input_size).

        Returns:
            torch.Tensor: The final predicted output, of shape
                          (batch_size, output_size).
        """
        # Get the batch size from the input tensor's shape.
        batch_size = x.size(0)

        # Initialize the hidden state h0 with zeros.
        # The shape is (num_layers * num_directions, batch_size, hidden_size).
        # Here, num_layers = 1 and num_directions = 1.
        # We use .to(x.device) to ensure h0 is on the same device as the input x.
        h0 = torch.zeros(1, batch_size, self.hidden_size).to(x.device)

        # Pass the input sequence x and the initial hidden state h0 through the RNN.
        # out: The output tensor containing the hidden state for every time step.
        #      Shape: (batch_size, sequence_length, hidden_size).
        # hn:  The final hidden state of the entire sequence.
        #      Shape: (num_layers * num_directions, batch_size, hidden_size).
        out, hn = self.rnn(x, h0)

        # Use only the output from the last time step to make the final prediction.
        # out[:, -1, :] selects:
        # - all samples in the batch
        # - the very last time step in the sequence
        # - all features in the hidden_size dimension
        # The resulting tensor has shape (batch_size, hidden_size).
        final_output = self.fc(out[:, -1, :])

        return final_output
